##### 1 XGBOOST - XGBOOST is used as an efficient implementation of Gradient Boosting which we recently learned about in class. XGBOOST uses decision trees as the weak learner, and combines multiple models. It is trained to try and correct the errors of the ensemble of trees before it. XGBOOST incorporates regularization techniques, which penalize complex models and helps prevent against overfitting, while also improving the model accuracy with the proper hyper-parameter tuning.

##### 2 The training methodology is as follows: first we load all neccesary packages to complete out training. Then we use a label encoder, because our dataset has [-1,1] label values (this sets the values to 0 and 1 respectively). Then we load our training dataset, and use the label encoder to transform the values. Next, we divide the data into train and test on a 60/40 split. Then, we establish our hyper-parameters for the model (we start out with the default values that you can find on any XGBOOSt documentation), using cross validation, we can loop through multiple values to find the best combination of hyperparameters. We then fit the XGBOOST model on the training data and hyper-parameters we set, based on the validation set performance. Finally, we make a prediction on the testing data and evaluate it.

##### 3 The hyperparameters are as follows: n_estimators, max_depth, reg_lambda, learning_rate, objective, missing. n_estimators is the number of trees you want the model to train on for each ensemble and its default value is 100. max_depth is the maximum depth of a tree, and increasing this value will make the model more complex and more likely to overfit and its default value is 6. reg_lambda is the L2 regularization term on weights, and increasing this value will make model more conservative and its default value is 1. learning_rate is step size shrinkage used in update to prevent overfitting and its default value is 0.3. objective is the learning objective of the model and its default value is "reg:squarederror". missing is the placeholder for missing values and its default value is np.nan. The final values are as follows: n_estimators = 100, max_depth = 3, reg_lambda = 3, learning_rate = 0.3, objective = "binary:logistic", missing = np.nan.



##### 4 The train accuracy I got from XGBOOST was 85.68%, hold-out accuracy was 84.85%, and test accuracy was 85.26%.

##### 5 Final Accuracy: 85.26%.

In [ ]:
# Problem 3 NN -- CMPSC 448 HW 4
# Aidan Vesci AJV5723

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

le = LabelEncoder()

# load data in LibSVM sparse data forma
X, y = load_svmlight_file("/content/a9a.txt")  # Training set
# X_test, y_test = load_svmlight_file("a9a.t") # Test set

le.fit(y)
y_transformed = le.transform(y)

# split data into train and test sets
seed = 6
test_size = 0.4
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=test_size, random_state=seed)


# fit model on training data
model = KNeighborsClassifier()

model.fit(X_train, y_train)

# make predictions for test data
y_pred = model.predict(X_test)
predictions = [round(value) for value in y_pred]
# evaluate predictions
accuracy = accuracy_score(y_test, predictions)
print(" Base Accuracy from Nearest Neighbors: %.2f%%" % (accuracy * 100.0))

 Base Accuracy from Nearest Neighbors: 82.69%


In [ ]:
# Problem 3 XGBOOST (train/hold-out/test + cross-validation) -- CMPSC 448 HW 4
# Aidan Vesci AJV5723

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import numpy as np

# Load training data
X, y = load_svmlight_file("/content/a9a.txt")  # Training set
le = LabelEncoder()
y = le.fit_transform(y)  # Switch labels from [-1, 1] to [0, 1]

# Hold-out split (train/validation)
seed = 6
test_size = 0.4
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=seed)

# Define model
model = XGBClassifier(objective='binary:logistic', missing = np.nan)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 150, 200, 250],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'reg_lambda': [0.5, 1, 2, 3, 4]
}

# Set up grid search with cross-validation
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,  # 5-fold cross-validation
    verbose=1,
    n_jobs=-1
)

# Run CV on training data
grid.fit(X_train, y_train)

# Get best model and parameters
best_model = grid.best_estimator_
print("\nBest Parameters:", grid.best_params_)

#### TRAIN ACCURACY
train_preds = best_model.predict(X_train)
train_acc = accuracy_score(y_train, train_preds)
print("Train Accuracy: {:.2f}%".format(train_acc * 100))

#### VALIDATION ACCURACY
val_preds = best_model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print("Validation Accuracy: {:.2f}%".format(val_acc * 100))

#### TEST ACCURACY

X_test_final, y_test_final = load_svmlight_file("/content/a9a.t")
y_test_final = le.transform(y_test_final)
test_preds = best_model.predict(X_test_final)
test_acc = accuracy_score(y_test_final, test_preds)
print("Test Accuracy: {:.2f}%".format(test_acc * 100))

Fitting 3 folds for each of 320 candidates, totalling 960 fits

Best Parameters: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 150, 'reg_lambda': 4}
Train Accuracy: 85.73%
Validation Accuracy: 84.83%
Test Accuracy: 85.24%
